In [1]:
import osmium
import geopandas as gpd
from shapely.geometry import Point

class DCHandler(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.rows = []

    def _keep(self, tags):
        return tags.get("telecom") == "data_center" or tags.get("building") == "data_center"

    def node(self, n):
        if self._keep(n.tags):
            self.rows.append({"name": n.tags.get("name"), "operator": n.tags.get("operator"),
                              "geometry": Point(n.location.lon, n.location.lat)})

    def area(self, a):
        if self._keep(a.tags):
            try:
                c = osmium.geom.WKBFactory().create_multipolygon(a)
                from shapely import wkb
                geom = wkb.loads(c, hex=True).centroid
                self.rows.append({"name": a.tags.get("name"), "operator": a.tags.get("operator"),
                                  "geometry": geom})
            except Exception:
                pass

h = DCHandler()
h.apply_file("/Users/maximedesjobert/Downloads/france-latest.osm.pbf", locations=True)

gdf = gpd.GeoDataFrame(h.rows, crs="EPSG:4326")
gdf.to_file("datacenters_points.geojson", driver="GeoJSON")
print(len(gdf), "data centers trouvés")

381 data centers trouvés


In [5]:
import geopandas as gpd
import pandas as pd

gdf = gpd.read_file("datacenters_points.geojson")
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y

df = pd.DataFrame(gdf.drop(columns="geometry"))
print(len(df), "data centers")
df

381 data centers


,name,operator,lon,lat
0,Association ReptiGo,None,6.011147,49.223391
1,SFR Netcenter Courbevoie,SFR,2.257787,48.903752
2,LyonIX 1 IN2P3,None,4.865745,45.782617
3,None,Cogent,-1.637296,48.109867
4,Telehouse Paris Voltaire,None,2.383304,48.856133
...,...,...,...,...
376,None,None,6.020041,46.251548
377,None,None,6.020099,46.251489
378,None,None,6.020145,46.251441
379,None,None,6.020202,46.251382


In [1]:
!pip install folium